In [1]:
!pip install numpy matplotlib
!pip install ipython

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [2]:
from quantum_error_corrections import (
    CNOT, H, I, P0, P1, S, T,
    U_N_qubits, U_one_gate, U_two_gates,
    X, Y, Z,
    apply_kraus,
    amplitude_damping_kraus,
    bit_flip_kraus,
    channel,
    controlled_gate,
    depolarizing_kraus,
    dm,
    evolve,
    fidelity_pure_rho,
    ket0, ket1,
    min_fidelity_bitflip_channel,
    min_fidelity_three_qubit_code,
    operator,
    pauli_kraus_channel,
    phase_damping_kraus,
    phase_flip_kraus,
    projectors,
    random_pure_state,
    rho,
    rhoToBlochVec,
    rotation_gate,
    sample_min_fidelity,
  improvement_condition,
    normalize,
    random_qubit_state,
    encode,
    majority_vote_correct,
    decode_to_single_qubit,
    monte_carlo_fidelity,
    pauli_string,
    apply_bitflips,

)

In [4]:
import numpy as np

# Gates
I = np.eye(2, dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)

# Helper: tensor product for multi-qubit operations
def U_N_qubits(ops):
    U = ops[0]
    for op in ops[1:]:
        U = np.kron(U, op)
    return U

# ----------------------------
# Noise models (Kraus operators)
# ----------------------------
def bit_flip_kraus(p):
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    return [np.sqrt(1 - p) * I, np.sqrt(p) * X]

def apply_kraus_noise(rho, kraus_ops, target, N):
    """Apply a single-qubit noise channel to target qubit in density matrix rho."""
    new_rho = np.zeros_like(rho)
    for E in kraus_ops:
        ops = [I] * N
        ops[target] = E
        E_full = U_N_qubits(ops)
        new_rho += E_full @ rho @ E_full.conj().T
    return new_rho

# ----------------------------
# DJA components
# ----------------------------
def initial_state_density(n):
    total = n + 1
    psi = np.zeros(2**total, dtype=complex)
    psi[1] = 1.0  # ancilla = 1
    return np.outer(psi, psi.conj())

def oracle_matrix(f, n):
    dim = 2 ** (n + 1)
    Uf = np.zeros((dim, dim), dtype=complex)
    for x in range(2**n):
        fx = f(x)
        for y in [0, 1]:
            in_idx = (x << 1) | y
            out_idx = (x << 1) | (y ^ fx)
            Uf[out_idx, in_idx] = 1
    return Uf

def measure_probs_first_n_density(rho, n):
    probs = np.zeros(2**n)
    for x in range(2**n):
        idx0 = 2*x
        idx1 = 2*x + 1
        probs[x] = np.real(rho[idx0, idx0] + rho[idx1, idx1])
    probs /= probs.sum()
    return probs

# ----------------------------
# Deutsch–Jozsa Algorithm with noise steps
# ----------------------------
def deutsch_jozsa(n, f, p=0.0, noise_stages=[]):
    """
    Deutsch–Jozsa Algorithm with optional bit-flip noise.
    noise_stages: list of stage numbers where noise is applied.
      1 = after initialization
      2 = after first Hadamards
      3 = after oracle
      4 = after final Hadamards
    """
    N = n + 1
    rho = initial_state_density(n)
    kraus_ops = bit_flip_kraus(p)

    # Stage 1: after initialization
    if 1 in noise_stages:
        rho = apply_kraus_noise(rho, kraus_ops, target=0, N=N)

    # Hadamard on all qubits
    H_full = U_N_qubits([H] * N)
    rho = H_full @ rho @ H_full.conj().T

    # Stage 2: after first Hadamards
    if 2 in noise_stages:
        rho = apply_kraus_noise(rho, kraus_ops, target=0, N=N)

    # Oracle
    Uf = oracle_matrix(f, n)
    rho = Uf @ rho @ Uf.conj().T

    # Stage 3: after oracle
    if 3 in noise_stages:
        rho = apply_kraus_noise(rho, kraus_ops, target=0, N=N)

    # Final Hadamards on input qubits only
    H_first_n = U_N_qubits([H] * n + [I])
    rho = H_first_n @ rho @ H_first_n.conj().T

    # Stage 4: after final Hadamards
    if 4 in noise_stages:
        rho = apply_kraus_noise(rho, kraus_ops, target=0, N=N)

    # Measure
    return measure_probs_first_n_density(rho, n)



    #Example 
def f_constant_0(x): return 0
def f_balanced_parity(x): return bin(x).count("1") % 2

n = 3

# Example: apply noise after each stage
for stage in [1, 2, 3, 4]:
    probs = deutsch_jozsa(n, f_balanced_parity, p=0.1, noise_stages=[stage])
    print(f"\nNoise applied at stage {stage}:")
    for i, pval in enumerate(probs):
        print(f"  |{i:03b}> : {pval:.4f}")


Noise applied at stage 1:
  |000> : 0.0000
  |001> : -0.0000
  |010> : -0.0000
  |011> : 0.1000
  |100> : -0.0000
  |101> : 0.0000
  |110> : 0.0000
  |111> : 0.9000

Noise applied at stage 2:
  |000> : 0.0000
  |001> : 0.0000
  |010> : 0.0000
  |011> : 0.0000
  |100> : 0.0000
  |101> : 0.0000
  |110> : 0.0000
  |111> : 1.0000

Noise applied at stage 3:
  |000> : 0.0000
  |001> : 0.0000
  |010> : 0.0000
  |011> : 0.0000
  |100> : 0.0000
  |101> : 0.0000
  |110> : 0.0000
  |111> : 1.0000

Noise applied at stage 4:
  |000> : 0.0000
  |001> : 0.0000
  |010> : 0.0000
  |011> : 0.1000
  |100> : 0.0000
  |101> : 0.0000
  |110> : 0.0000
  |111> : 0.9000
